In [ ]:
#CODE 
# still double checking that everything makes sense mathematically and matches the method mentioned in "Liu & Kirshbaum 2026"

"""
Spectral solver for perturbation-pressure decomposition (manuscript Appendix A2-A5).

Method:
  1. Vertically interpolate the RHS forcing field onto a uniform grid (dz = 50 m).
  2. 2D horizontal Fourier transform of the doubly periodic RHS at each level,
     reducing the 3D Poisson equation to an ODE in z per wavenumber (k, l):
         d2/dz2 piHat - (k^2 + l^2) piHat = rHat(k, l, z)          (A4)
  3. Solve the Nz x Nz linear system per wavenumber:
       - interior rows: centered differencing of d2/dz2
       - top BC: zero-gradient (Neumann), one-sided differencing
       - bottom BC: integral condition (A5), one-sided derivative + discrete sums
  4. Skip (k, l) = (0, 0) (singular mode -> decomposed field has zero mean).
  5. Inverse 2D FFT at each level to recover pi'(x, y, z) in physical space.

Naming conventions: variables camelCase, functions PascalCase,
function returns wrapped in square brackets.
"""

import numpy as np
from tqdm import tqdm

# numpy >=2.0 renamed trapz -> trapezoid; support either.
_TrapezoidFunc = getattr(np, "trapezoid", None) or np.trapz


# ----------------------------------------------------------------------
# Helper functions (repeated tasks)
# ----------------------------------------------------------------------

def InterpolateColumn(zOld, columnOld, zNew):
    """Linearly interpolate one vertical column onto the new uniform grid."""
    columnNew = np.interp(zNew, zOld, columnOld)
    return [columnNew]


def BuildWavenumbers(nx, ny, lx, ly):
    """Angular wavenumbers k (x-direction) and l (y-direction) matching np.fft.fft2 layout."""
    kVals = 2.0 * np.pi * np.fft.fftfreq(nx, d=lx / nx)
    lVals = 2.0 * np.pi * np.fft.fftfreq(ny, d=ly / ny)
    return [kVals, lVals]


def BuildInteriorRow(matrix, rowIndex, kl2, dz):
    """Fill one interior row: centered differencing of d2/dz2 minus (k^2+l^2)."""
    invDz2 = 1.0 / dz**2
    matrix[rowIndex, rowIndex - 1] = invDz2
    matrix[rowIndex, rowIndex] = -2.0 * invDz2 - kl2
    matrix[rowIndex, rowIndex + 1] = invDz2
    return [matrix]


def BuildTopRow(matrix, nz, dz):
    """Top BC: zero-gradient (Neumann), one-sided (backward) differencing at z = zt."""
    matrix[nz - 1, nz - 1] = 1.0 / dz
    matrix[nz - 1, nz - 2] = -1.0 / dz
    return [matrix]


def BuildBottomRow(matrix, nz, kl2, dz):
    """
    Bottom BC: integral condition (A5),
       -dpi/dz|_0  -  (k^2+l^2) * Integral(pi dz)  =  Integral(R dz),
    with the top-derivative term already zero from the top BC.
    One-sided (forward) differencing for dpi/dz|_0; integrals as discrete sums
    (trapezoidal weights).
    """
    # one-sided derivative at the bottom: -(pi[1] - pi[0]) / dz
    matrix[0, 0] += 1.0 / dz
    matrix[0, 1] += -1.0 / dz
    # -(k^2+l^2) * trapezoidal sum of pi over the column
    weights = np.full(nz, dz)
    weights[0] = 0.5 * dz
    weights[nz - 1] = 0.5 * dz
    matrix[0, :] += -kl2 * weights
    return [matrix]


def IntegrateColumn(columnValues, dz):
    """Discrete (trapezoidal) vertical integral of one column."""
    integralValue = _TrapezoidFunc(columnValues, dx=dz)
    return [integralValue]


def SolveOneWavenumber(rHatColumn, kl2, dz):
    """
    Assemble and solve the Nz x Nz system for one (k, l) pair.
    Rows: 0 = bottom integral BC, 1..Nz-2 = interior centered differencing,
    Nz-1 = top Neumann BC.
    """
    nz = rHatColumn.size
    matrix = np.zeros((nz, nz), dtype=complex)
    rhsVector = np.zeros(nz, dtype=complex)

    # interior rows
    for rowIndex in range(1, nz - 1):
        [matrix] = BuildInteriorRow(matrix, rowIndex, kl2, dz)
        rhsVector[rowIndex] = rHatColumn[rowIndex]

    # top row: Neumann, RHS = 0
    [matrix] = BuildTopRow(matrix, nz, dz)
    rhsVector[nz - 1] = 0.0

    # bottom row: integral condition, RHS = vertical integral of rHat
    [matrix] = BuildBottomRow(matrix, nz, kl2, dz)
    [rhsVector[0]] = IntegrateColumn(rHatColumn, dz)

    piHatColumn = np.linalg.solve(matrix, rhsVector)
    return [piHatColumn]


# ----------------------------------------------------------------------
# Main pipeline
# ----------------------------------------------------------------------

def InterpolateRhsVertically(rhsField, zOld, dz=50.0):
    """
    Step 1: interpolate rhsField(nzOld, ny, nx) onto a uniform vertical grid
    with spacing dz (default 50 m), column by column.
    """
    zNew = np.arange(zOld[0], zOld[-1] + 0.5 * dz, dz)
    nzNew = zNew.size
    nzOld, ny, nx = rhsField.shape
    rhsUniform = np.zeros((nzNew, ny, nx))
    for jIndex in range(ny):
        for iIndex in range(nx):
            [rhsUniform[:, jIndex, iIndex]] = InterpolateColumn(
                zOld, rhsField[:, jIndex, iIndex], zNew)
    return [rhsUniform, zNew]


def TransformRhsHorizontally(rhsUniform):
    """Step 2: 2D FFT of the doubly periodic RHS at each vertical level."""
    rhsHat = np.fft.fft2(rhsUniform, axes=(1, 2))
    return [rhsHat]


def SolveAllWavenumbers(rhsHat, kVals, lVals, dz):
    """
    Step 3-4: loop over every (k, l), solve the ODE system, skipping the
    singular central wavenumber (k, l) = (0, 0), whose coefficients are
    left at zero (=> decomposed field has zero horizontal mean).
    """
    nz, ny, nx = rhsHat.shape
    piHat = np.zeros((nz, ny, nx), dtype=complex)
    for jIndex in tqdm(range(ny), desc="wavenumber rows (l)"):
        for iIndex in range(nx):
            if iIndex == 0 and jIndex == 0:
                continue
            kl2 = kVals[iIndex]**2 + lVals[jIndex]**2
            [piHat[:, jIndex, iIndex]] = SolveOneWavenumber(
                rhsHat[:, jIndex, iIndex], kl2, dz)
    return [piHat]


def InverseTransform(piHat):
    """Step 5: inverse 2D FFT at each level to recover pi'(x, y, z)."""
    piPhysical = np.real(np.fft.ifft2(piHat, axes=(1, 2)))
    return [piPhysical]


def ComputeDiscreteLaplacian(piField, kVals, lVals, dz):
    """
    Recompute the Laplacian of a solved field using the SAME discretization
    as the solver: horizontal part via spectral multiplication by -(k^2+l^2),
    vertical part via centered differencing (interior points only).
    Used to verify self-consistency between the solved field and the RHS
    it was solved for (independent check from comparing against a known
    manufactured solution).
    """
    nz, ny, nx = piField.shape
    piHat = np.fft.fft2(piField, axes=(1, 2))

    laplacianHat = np.zeros_like(piHat)
    for jIndex in range(ny):
        for iIndex in range(nx):
            kl2 = kVals[iIndex]**2 + lVals[jIndex]**2
            horizPart = -kl2 * piHat[:, jIndex, iIndex]
            vertPart = np.zeros(nz, dtype=complex)
            vertPart[1:-1] = (piHat[2:, jIndex, iIndex]
                               - 2.0 * piHat[1:-1, jIndex, iIndex]
                               + piHat[:-2, jIndex, iIndex]) / dz**2
            laplacianHat[:, jIndex, iIndex] = horizPart + vertPart

    laplacianField = np.real(np.fft.ifft2(laplacianHat, axes=(1, 2)))
    return [laplacianField]


def DecomposePressureComponent(rhsField, zOld, lx, ly, dz=50.0):
    """
    Full pipeline for one pressure component (e.g. pi'_b with R = d(sigma0*B)/dz).
    rhsField : (nzOld, ny, nx) forcing on the model's native vertical grid
    zOld     : native vertical levels (m)
    lx, ly   : horizontal domain lengths (m)
    Returns the decomposed pressure field on the uniform grid, plus that grid.
    """
    [rhsUniform, zNew] = InterpolateRhsVertically(rhsField, zOld, dz)
    [rhsHat] = TransformRhsHorizontally(rhsUniform)
    nx = rhsUniform.shape[2]
    ny = rhsUniform.shape[1]
    [kVals, lVals] = BuildWavenumbers(nx, ny, lx, ly)
    [piHat] = SolveAllWavenumbers(rhsHat, kVals, lVals, dz)
    [piField] = InverseTransform(piHat)
    return [piField, zNew]


def PlotComparisonPanels(rhsSlice, piSlice, laplacianSlice, xVals, yVals, zLevelLabel,
                          matchDiffColorToRhs=False):
    """
    4-panel comparison plot at one horizontal (x,y) level:
      1. original RHS function
      2. solved field (pi')
      3. Laplacian of the solved field (recomputed)
      4. difference: panel 1 minus panel 3 (should be ~0 at interior levels)

    matchDiffColorToRhs : if True, panel 4 uses the same color limits as
        panel 1 (useful for visually confirming the difference is small
        relative to the original signal). If False (default), panel 4
        gets its own independent color scale.
    """
    import matplotlib.pyplot as plt
    diffSlice = rhsSlice - laplacianSlice
    panelData = [rhsSlice, piSlice, laplacianSlice, diffSlice]
    panelTitles = ["Original RHS", "Solved field (pi')",
                   "Laplacian(solution)", "Difference (RHS - Laplacian)"]
    fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)

    firstVLim = np.max(np.abs(panelData[0])) if np.max(np.abs(panelData[0])) > 0 else 1.0

    for axIndex, ax in enumerate(axes):
        data = panelData[axIndex]
        if axIndex == 3 and matchDiffColorToRhs:
            vLim = firstVLim
        else:
            vLim = np.max(np.abs(data)) if np.max(np.abs(data)) > 0 else 1.0
        mesh = ax.pcolormesh(xVals/1e3, yVals/1e3, data, cmap="RdBu_r",
                              vmin=-vLim, vmax=vLim, shading="auto")
        ax.set_title(panelTitles[axIndex])
        ax.set_xlabel("x (m)")
        if axIndex == 0:
            ax.set_ylabel("y (m)")
        ax.set_aspect("equal")
        fig.colorbar(mesh, ax=ax, shrink=0.8)
    fig.suptitle(f"Field comparison at {zLevelLabel}")

    return [fig]

In [ ]:
# 2D Test (2D Dirac Delta)
# ----------------------------------------------------------------------

# domain
lx, ly = 20000.0, 20000.0          # 20 km x 20 km
nx, ny = 32, 32
dz = 50.0
zTop = 5000.0
zModel = np.linspace(0.0, zTop, 41)  # non-uniform-ish native grid stand-in
xVals = np.arange(nx) * lx / nx
yVals = np.arange(ny) * ly / ny
x3, y3 = np.meshgrid(xVals, yVals)   # (ny, nx)

# --- 2D discrete delta-function RHS (point source, constant in z) ---
dx = lx / nx
dy = ly / ny

iCenter = nx // 2
jCenter = ny // 2

deltaHoriz = np.zeros((ny, nx))
deltaHoriz[jCenter, iCenter] = 1.0 / (dx * dy)   # normalized so sum*dx*dy = 1

rhsTrue = np.zeros((zModel.size, ny, nx))
for n, zVal in enumerate(zModel):
    rhsTrue[n] = deltaHoriz   # same point source at every level

[piSolved, zNew] = DecomposePressureComponent(rhsTrue, zModel, lx, ly, dz)

# interpolate truth to solver grid for comparison
piTrueUniform = np.zeros_like(piSolved)
for jIndex in range(ny):
    for iIndex in range(nx):
        [piTrueUniform[:, jIndex, iIndex]] = InterpolateColumn(
            zModel, piTrue[:, jIndex, iIndex], zNew)

errorMax = np.max(np.abs(piSolved - piTrueUniform))
ampMax = np.max(np.abs(piTrueUniform))
print(f"grid: {nx} x {ny} x {zNew.size}   dz = {dz} m")
print(f"max |error|          : {errorMax:.4e}")
print(f"max |true field|     : {ampMax:.4e}")
print(f"relative max error   : {errorMax / ampMax:.4e}")
print(f"solved-field mean    : {piSolved.mean():.3e}  (should be ~0, (0,0) mode excluded)")

# --- Laplacian self-consistency check ---
# interpolate the true RHS onto the same uniform grid for comparison
rhsUniform = np.zeros_like(piSolved)
for jIndex in range(ny):
    for iIndex in range(nx):
        [rhsUniform[:, jIndex, iIndex]] = InterpolateColumn(
            zModel, rhsTrue[:, jIndex, iIndex], zNew)
# remove the (0,0) horizontal mean from the RHS too, since the solver
# never sees/solves that mode (solved field has zero mean by construction)
rhsUniform -= rhsUniform.mean(axis=(1, 2), keepdims=True)

[kVals, lVals] = BuildWavenumbers(nx, ny, lx, ly)
[laplacianOfSolved] = ComputeDiscreteLaplacian(piSolved, kVals, lVals, dz)

# compare only interior vertical levels: the discrete Laplacian here
# uses simple centered differencing with no BC modification, so the
# top/bottom rows are not expected to match (those rows enforce the
# Neumann / integral conditions instead of the interior PDE).
interiorSlice = slice(1, -1)
residual = laplacianOfSolved[interiorSlice] - rhsUniform[interiorSlice]
residualMax = np.max(np.abs(residual))
rhsAmpMax = np.max(np.abs(rhsUniform[interiorSlice]))
print(f"\n--- Laplacian(piSolved) vs RHS, interior levels only ---")
print(f"max |Laplacian - RHS| : {residualMax:.4e}")
print(f"max |RHS|             : {rhsAmpMax:.4e}")
print(f"relative residual     : {residualMax / rhsAmpMax:.4e}")

# --- 4-panel plot: pick a level with real signal, not a node of the ---
# --- vertical structure (avoid comparing noise against noise)       ---
levelAmplitudes = np.max(np.abs(rhsUniform[1:-1]), axis=(1, 2))  # interior levels only
levelIndex = 1 + np.argmax(levelAmplitudes)  # +1 to offset the interior slice
zLevelLabel = f"z = {zNew[levelIndex]:.0f} m"

PlotComparisonPanels(rhsUniform[levelIndex], piSolved[levelIndex], laplacianOfSolved[levelIndex],
                     xVals, yVals, zLevelLabel,
                     matchDiffColorToRhs=False)

In [ ]:
# 2D Test (2D Gaussian)
# ----------------------------------------------------------------------

# domain
lx, ly = 20000.0, 20000.0          # 20 km x 20 km
nx, ny = 32, 32
dz = 50.0
zTop = 5000.0
zModel = np.linspace(0.0, zTop, 41)  # non-uniform-ish native grid stand-in
xVals = np.arange(nx) * lx / nx
yVals = np.arange(ny) * ly / ny
x3, y3 = np.meshgrid(xVals, yVals)   # (ny, nx)

# --- 2D Gaussian RHS (localized bump, constant in z) ---
x0, y0 = lx / 2.0, ly / 2.0       # center of domain
sigmaX, sigmaY = 2000.0, 2000.0   # Gaussian width (m)
amplitude = 1.0e-6                # scaled roughly to match earlier RHS magnitude

gaussianHoriz = amplitude * np.exp(
    -((x3 - x0)**2) / (2.0 * sigmaX**2)
    -((y3 - y0)**2) / (2.0 * sigmaY**2)
)

rhsTrue = np.zeros((zModel.size, ny, nx))
for n, zVal in enumerate(zModel):
    rhsTrue[n] = gaussianHoriz   # same horizontal Gaussian at every level

[piSolved, zNew] = DecomposePressureComponent(rhsTrue, zModel, lx, ly, dz)

# interpolate truth to solver grid for comparison
piTrueUniform = np.zeros_like(piSolved)
for jIndex in range(ny):
    for iIndex in range(nx):
        [piTrueUniform[:, jIndex, iIndex]] = InterpolateColumn(
            zModel, piTrue[:, jIndex, iIndex], zNew)

errorMax = np.max(np.abs(piSolved - piTrueUniform))
ampMax = np.max(np.abs(piTrueUniform))
print(f"grid: {nx} x {ny} x {zNew.size}   dz = {dz} m")
print(f"max |error|          : {errorMax:.4e}")
print(f"max |true field|     : {ampMax:.4e}")
print(f"relative max error   : {errorMax / ampMax:.4e}")
print(f"solved-field mean    : {piSolved.mean():.3e}  (should be ~0, (0,0) mode excluded)")

# --- Laplacian self-consistency check ---
# interpolate the true RHS onto the same uniform grid for comparison
rhsUniform = np.zeros_like(piSolved)
for jIndex in range(ny):
    for iIndex in range(nx):
        [rhsUniform[:, jIndex, iIndex]] = InterpolateColumn(
            zModel, rhsTrue[:, jIndex, iIndex], zNew)
# remove the (0,0) horizontal mean from the RHS too, since the solver
# never sees/solves that mode (solved field has zero mean by construction)
rhsUniform -= rhsUniform.mean(axis=(1, 2), keepdims=True)

[kVals, lVals] = BuildWavenumbers(nx, ny, lx, ly)
[laplacianOfSolved] = ComputeDiscreteLaplacian(piSolved, kVals, lVals, dz)

# compare only interior vertical levels: the discrete Laplacian here
# uses simple centered differencing with no BC modification, so the
# top/bottom rows are not expected to match (those rows enforce the
# Neumann / integral conditions instead of the interior PDE).
interiorSlice = slice(1, -1)
residual = laplacianOfSolved[interiorSlice] - rhsUniform[interiorSlice]
residualMax = np.max(np.abs(residual))
rhsAmpMax = np.max(np.abs(rhsUniform[interiorSlice]))
print(f"\n--- Laplacian(piSolved) vs RHS, interior levels only ---")
print(f"max |Laplacian - RHS| : {residualMax:.4e}")
print(f"max |RHS|             : {rhsAmpMax:.4e}")
print(f"relative residual     : {residualMax / rhsAmpMax:.4e}")

# --- 4-panel plot: pick a level with real signal, not a node of the ---
# --- vertical structure (avoid comparing noise against noise)       ---
levelAmplitudes = np.max(np.abs(rhsUniform[1:-1]), axis=(1, 2))  # interior levels only
levelIndex = 1 + np.argmax(levelAmplitudes)  # +1 to offset the interior slice
zLevelLabel = f"z = {zNew[levelIndex]:.0f} m"

PlotComparisonPanels(rhsUniform[levelIndex], piSolved[levelIndex], laplacianOfSolved[levelIndex],
                     xVals, yVals, zLevelLabel,
                     matchDiffColorToRhs=False)

In [ ]:
# 2D Test (Cosine Wave)
# ----------------------------------------------------------------------


# domain
lx, ly = 20000.0, 20000.0          # 20 km x 20 km
nx, ny = 32, 32
dz = 50.0
zTop = 5000.0
zModel = np.linspace(0.0, zTop, 41)  # non-uniform-ish native grid stand-in

xVals = np.arange(nx) * lx / nx
yVals = np.arange(ny) * ly / ny
x3, y3 = np.meshgrid(xVals, yVals)   # (ny, nx)

# manufactured pi: single horizontal mode * vertical structure with
# dpi/dz = 0 at BOTH boundaries (cos(pi z/zt)) so BCs are consistent
kTest = 2.0 * np.pi * 2 / lx
lTest = 2.0 * np.pi * 1 / ly
mTest = np.pi / zTop
horiz = np.cos(kTest * x3 + lTest * y3)

piTrue = np.zeros((zModel.size, ny, nx))
rhsTrue = np.zeros((zModel.size, ny, nx))
for n, zVal in enumerate(zModel):
    vert = np.cos(mTest * zVal)
    piTrue[n] = horiz * vert
    # lap(pi) = -(k^2 + l^2 + m^2) pi
    rhsTrue[n] = -(kTest**2 + lTest**2 + mTest**2) * horiz * vert

[piSolved, zNew] = DecomposePressureComponent(rhsTrue, zModel, lx, ly, dz)

# interpolate truth to solver grid for comparison
piTrueUniform = np.zeros_like(piSolved)
for jIndex in range(ny):
    for iIndex in range(nx):
        [piTrueUniform[:, jIndex, iIndex]] = InterpolateColumn(
            zModel, piTrue[:, jIndex, iIndex], zNew)

errorMax = np.max(np.abs(piSolved - piTrueUniform))
ampMax = np.max(np.abs(piTrueUniform))
print(f"grid: {nx} x {ny} x {zNew.size}   dz = {dz} m")
print(f"max |error|          : {errorMax:.4e}")
print(f"max |true field|     : {ampMax:.4e}")
print(f"relative max error   : {errorMax / ampMax:.4e}")
print(f"solved-field mean    : {piSolved.mean():.3e}  (should be ~0, (0,0) mode excluded)")

# --- Laplacian self-consistency check ---
# interpolate the true RHS onto the same uniform grid for comparison
rhsUniform = np.zeros_like(piSolved)
for jIndex in range(ny):
    for iIndex in range(nx):
        [rhsUniform[:, jIndex, iIndex]] = InterpolateColumn(
            zModel, rhsTrue[:, jIndex, iIndex], zNew)
# remove the (0,0) horizontal mean from the RHS too, since the solver
# never sees/solves that mode (solved field has zero mean by construction)
rhsUniform -= rhsUniform.mean(axis=(1, 2), keepdims=True)

[kVals, lVals] = BuildWavenumbers(nx, ny, lx, ly)
[laplacianOfSolved] = ComputeDiscreteLaplacian(piSolved, kVals, lVals, dz)

# compare only interior vertical levels: the discrete Laplacian here
# uses simple centered differencing with no BC modification, so the
# top/bottom rows are not expected to match (those rows enforce the
# Neumann / integral conditions instead of the interior PDE).
interiorSlice = slice(1, -1)
residual = laplacianOfSolved[interiorSlice] - rhsUniform[interiorSlice]
residualMax = np.max(np.abs(residual))
rhsAmpMax = np.max(np.abs(rhsUniform[interiorSlice]))
print(f"\n--- Laplacian(piSolved) vs RHS, interior levels only ---")
print(f"max |Laplacian - RHS| : {residualMax:.4e}")
print(f"max |RHS|             : {rhsAmpMax:.4e}")
print(f"relative residual     : {residualMax / rhsAmpMax:.4e}")

# --- 4-panel plot: pick a level with real signal, not a node of the ---
# --- vertical structure (avoid comparing noise against noise)       ---
levelAmplitudes = np.max(np.abs(rhsUniform[1:-1]), axis=(1, 2))  # interior levels only
levelIndex = 1 + np.argmax(levelAmplitudes)  # +1 to offset the interior slice
zLevelLabel = f"z = {zNew[levelIndex]:.0f} m"

PlotComparisonPanels(rhsUniform[levelIndex], piSolved[levelIndex], laplacianOfSolved[levelIndex],
                     xVals, yVals, zLevelLabel,
                     matchDiffColorToRhs=False)

In [ ]:
# 3D Test Loading in real data to test
# ----------------------------------------------------------------------

import os,sys

#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

####################################
#LOADING CLASSES

#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="Buoyancy",
                                dtype='float32')

####################################
#FUNCTIONS
def ComputeBaseStateThetaRho(th_rho):
    """Base-state (horizontal-mean) density potential temperature at each level."""
    th_rho0 = th_rho.mean(axis=(1, 2))
    return [th_rho0]


def ComputeSigma0(th_rho, cp=1004.0):
    """
    sigma0 = (cp * theta_rho0)^-1, where theta_rho0 is the horizontal-mean
    (base-state) density potential temperature at each level.
    th_rho : 3D array, shape (nz, ny, nx)
    """
    [th_rho0] = ComputeBaseStateThetaRho(th_rho)
    sigma0 = 1.0 / (cp * th_rho0)
    return [sigma0]

def Ddz(field, zModel):
    """
    Vertical derivative of a 3D field along axis 0 (z), using centered
    differencing in the interior and one-sided differencing at the
    top/bottom (via np.gradient).
    field  : 3D array, shape (nz, ny, nx)
    zModel : 1D array of vertical levels, shape (nz,)
    """
    derivative = np.gradient(field, zModel, axis=0)
    return [derivative]

def MakePlot(array,
            nLevels=21,
            zLevels=ModelData.zh*1e3,
            vMax=1.5e-9):
    
    xh=ModelData.xh-ModelData.xh[0]
    array_zx=array[:, 100]
    
    if vMax is None:
        vMax=np.max(array_zx)
    levels=np.linspace(-vMax,vMax,nLevels)
    # levels=nLevels
    plt.contourf(xh,zLevels/1e3,array_zx,levels=levels,cmap='RdBu_r');plt.colorbar()
    plt.ylim(0,6)
def MakePlotOnAxis(ax, array, zLevels, title, nLevels=21, vMax=None):
    """Same logic as MakePlot, but targets a given axis for subplotting."""
    xh = ModelData.xh - ModelData.xh[0]
    arrayZx = array[:, 100]

    if vMax is None:
        vMax = np.max(np.abs(arrayZx))
    levels = np.linspace(-vMax, vMax, nLevels)

    mesh = ax.contourf(xh, zLevels/1e3, arrayZx, levels=levels, cmap='RdBu_r')
    ax.set_ylim(0, 6)
    ax.set_title(title)
    plt.colorbar(mesh, ax=ax, shrink=0.8)
    

####################################
#LOADING DATA

t=100
B=CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName='buoyancy2')
th_rho=CallVariable(ModelData,DataManager,ModelData.timeStrings[t],variableName='theta_v')

####################################
#CALCULATE RIGHT


[sigma0]=ComputeSigma0(th_rho)

f = sigma0[:,np.newaxis,np.newaxis]*B

[RHS] = Ddz(f, ModelData.zh*1e3)
# MakePlot(RHS)
# [rhsUniform, zNew] = InterpolateRhsVertically(RHS, ModelData.zh*1e3, dz=50.0)
# MakePlot(rhsUniform,zLevels=zNew)

[pi_b_prime, zNew] = DecomposePressureComponent(RHS, ModelData.zh*1e3, 
                                                lx=ModelData.Nxh*ModelData.dx, 
                                                ly=ModelData.Nyh*ModelData.dy, dz=50)

# --- get RHS onto the same grid pi_b_prime lives on ---
# [rhsUniform, zNew] = InterpolateRhsVertically(RHS, ModelData.zh*1e3, dz=50.0)

# --- recompute Laplacian of the solved field ---
nx = ModelData.Nxh
ny = ModelData.Nyh
lx = ModelData.Nxh * ModelData.dx
ly = ModelData.Nyh * ModelData.dy
[kVals, lVals] = BuildWavenumbers(nx, ny, lx, ly)
[laplacianOfPiB] = ComputeDiscreteLaplacian(pi_b_prime, kVals, lVals, dz=50.0)

# --- difference: RHS minus Laplacian(pi_b_prime) ---
diffField = rhsUniform - laplacianOfPiB

# --- plotting ---
fig, axes = plt.subplots(1, 4, figsize=(22, 5), constrained_layout=True)

MakePlotOnAxis(axes[0], rhsUniform, zNew, "RHS")
MakePlotOnAxis(axes[1], pi_b_prime, zNew, "pi_b_prime")
MakePlotOnAxis(axes[2], laplacianOfPiB, zNew, "Laplacian(pi_b_prime)")
MakePlotOnAxis(axes[3], diffField, zNew, "RHS - Laplacian")